# 07 · Export for the app

Writes the two files the app needs, into `export/` (committed, so the app can take them without running the notebooks):

| File | Contents | Replaces in the app |
| --- | --- | --- |
| `export/presets.json` | The presets chosen in notebook 05: a plain-language name and description, and the rules in the app's rule format with plain-language titles | the rules the old questionnaire generated |
| `export/onboarding.json` | The onboarding tree: questions with plain answer labels, branches, and the preset at each leaf | `public/mock/questionnaire.json` (seven questions) |

Then it **checks the exported files themselves**: applying the exported rules the way the app does gives exactly each
preset's decisions on all 6,272 items, every term is a core term, conditions sit only where the taxonomy allows
them, and walking the tree reaches a preset for every household.

**What the app needs before it can use these** (see "For the app" at the end): a `condition` on allow rules, a
`preset` rule origin, and a walker for the tree.

In [1]:
import json
from datetime import date
from pathlib import Path
import numpy as np, pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
EXPORT = ROOT / "export"
items = pd.read_csv(DATA / "items.csv")
D = np.load(DATA / "decisions.npy")
answers = pd.read_csv(DATA / "answers.csv", dtype=str)
QUESTIONS = {q["id"]: q for q in json.loads((DATA / "questions.json").read_text())}
tree = json.loads((DATA / "onboarding_tree.json").read_text())
K = tree["k"]
presets = json.loads((DATA / tree["presets_file"]).read_text())
preset_items = np.load(DATA / "presets" / f"preset_items_k{K}.npy")
FIELDS = ["data_type", "purpose", "action", "source", "handling_context"]
TERMS = {f: sorted(items[f].unique()) for f in FIELDS}
real = items.in_corpus.to_numpy()
ORDINAL = np.array([0.0, 1.0, 0.5, 0.5, 0.5])
print(f"exporting k = {K} presets and the depth-{tree['depth']} tree (avg {tree['questions_avg']} questions)")

exporting k = 8 presets and the depth-5 tree (avg 4.86 questions)


## Plain language

Every core term gets a short label households can read. These are working labels for review.

In [2]:
PLAIN = {
    "ppd:identifierData": "account and identity details", "ppd:profileData": "profile and preferences",
    "ppd:sensorData": "sensor readings", "ppd:contentData": "video, photos and sound", "ppd:locationData": "location",
    "ppd:deviceAndNetworkData": "device and network details", "ppd:usageData": "how you use your devices",

    "ppd:coreFunctionality": "making features work", "ppd:personalization": "personalizing what you see",
    "ppd:communicationsAndNotifications": "alerts and messages", "ppd:security": "security",
    "ppd:diagnosticsAndMaintenance": "fixing problems", "ppd:analyticsAndImprovement": "improving products",
    "ppd:legalCompliance": "legal obligations", "ppd:advertisingAndMarketing": "advertising",

    "ppd:collection": "collected", "ppd:use": "used", "ppd:transfer": "sent on", "ppd:inference": "used to work out new information",

    "ppd:userProvided": "you entered", "ppd:participantObserved": "the device observes", "ppd:participantGenerated": "the device generates",
    "ppd:householdProvided": "other devices in your home provide", "ppd:vendorProvided": "the maker provides",
    "ppd:thirdPartyProvided": "other companies provide", "ppd:derivedFromPriorData": "worked out from earlier data",

    "ppd:householdContext": "staying in your home", "ppd:vendorContext": "going to the device maker",
    "ppd:thirdPartyContext": "going to other companies", "ppd:publicAudience": "made public",
}
SHORT = {   # for preset names
    "ppd:advertisingAndMarketing": "ads", "ppd:analyticsAndImprovement": "product improvement", "ppd:personalization": "personalization",
    "ppd:communicationsAndNotifications": "alerts", "ppd:diagnosticsAndMaintenance": "fixing problems", "ppd:coreFunctionality": "features",
    "ppd:security": "security", "ppd:legalCompliance": "legal duties",
    "ppd:thirdPartyContext": "other companies", "ppd:publicAudience": "anything public", "ppd:vendorContext": "the maker", "ppd:householdContext": "keeping it home",
    "ppd:contentData": "video", "ppd:locationData": "location", "ppd:identifierData": "identity", "ppd:profileData": "profiles",
    "ppd:sensorData": "sensor data", "ppd:usageData": "usage data", "ppd:deviceAndNetworkData": "device details",
}
EFFECT_TEXT = {0: "Never", 1: "Allow", 2: "Allow only if not kept", 3: "Allow only if processed in your home", 4: "Allow only if processed on the device"}
OPTION_LABEL = {"never": "Never", "ask": "It depends", "fine": "That's fine", "features": "Only for features I use", "ok": "That's fine",
                "briefly": "Only briefly", "any": "No limit", "no": "No", "with_limits": "Only with limits", "yes": "Yes",
                "1": "1 · Not at all", "2": "2", "3": "3", "4": "4", "5": "5 · A lot"}
assert set(PLAIN) == {t for f in FIELDS for t in TERMS[f]}, "every core term needs a plain label" 

## Rule titles

A rule's scope lists the terms it covers per field; a field covering most of its terms reads better as "any … except".
Example: *Never: video, photos and sound · going to other companies or made public · for anything except making
features work*.

In [3]:
def terms_text(field, terms):
    everything = TERMS[field]
    if len(terms) == len(everything):
        return None
    missing = [t for t in everything if t not in terms]
    if field == "handling_context":
        if set(terms) == set(everything) - {"ppd:householdContext"}:
            return "leaving your home"
        return " or ".join(PLAIN[t] for t in terms)
    words = {"data_type": ("", "any data except "), "purpose": ("for ", "for anything except "),
             "action": ("when ", "except when "), "source": ("that ", "unless ")}[field]
    if len(missing) < len(terms):
        return words[1] + ", ".join(PLAIN[t] for t in missing) if field != "action" else "except when " + " or ".join(PLAIN[t] for t in missing)
    return words[0] + (" or ".join(PLAIN[t] for t in terms) if field in ("action", "source") else ", ".join(PLAIN[t] for t in terms))

def decision_of(rule):
    if rule["effect"] == "dont_allow":
        return 0
    return {None: 1, "retention=ppd:ephemeral": 2, "processing_boundary=ppd:inHomeOnly": 3, "processing_boundary=ppd:onDeviceOnly": 4}[rule["condition"]]

def title(rule):
    scope = {f: rule["scope"].get(f, TERMS[f]) for f in FIELDS}
    order = ["data_type", "handling_context", "purpose", "action", "source"]
    parts = [p for f in order if (p := terms_text(f, scope[f]))]
    return f"{EFFECT_TEXT[decision_of(rule)]}: " + (" · ".join(parts) if parts else "everything else")

## Preset names and descriptions

A name is a CableLabs tier (*Privacy-tightened*, *Balanced*, *Cloud-assisted*, the names of CableLabs' example
policies) chosen by how much of all flows the preset accepts, followed by a short name chosen by hand in `PRESET_NAMES`
from where the preset is most and least accepting compared with all households. The automatic name is the fallback for
any preset without one.

In [4]:
# Reviewed names for the chosen k = 8 presets, each based on where the preset is strictest and most open
# (acceptance per purpose, recipient and data type compared with all households). Automatic names stay as a fallback.
PRESET_NAMES = {
    "k8-p0": "Keep it at home",                  # strictest overall; home always fine, other companies and public rarely
    "k8-p1": "No personalization",               # refuses personalization, otherwise middling
    "k8-p2": "Device maker only",                # trusts the maker, wary of other companies and anything public
    "k8-p3": "Ads OK, no analytics",             # refuses product analytics, accepts ads
    "k8-p4": "Ads and analytics OK",             # most accepting overall
    "k8-p5": "No ads",                           # refuses ads, accepts nearly everything else
    "k8-p6": "No ads, no analytics",             # refuses ads and product analytics
    "k8-p7": "No ads, careful with sharing",     # refuses ads, cautious with other companies and public
}
overall = ORDINAL[D].mean(axis=0)

# The first part of every name is one of CableLabs' example-policy tiers (habanero-ppd-gateway policy_examples:
# privacy-tightened, balanced, cloud-assisted), chosen by the share of all flows the preset accepts.
def tier(i):
    level = ORDINAL[preset_items[i]].mean()
    return "Privacy-tightened" if level < 0.55 else "Cloud-assisted" if level > 0.75 else "Balanced"

def describe(i):
    acc = ORDINAL[preset_items[i]]
    deltas = {}
    for f in ["purpose", "handling_context", "data_type"]:
        g = pd.Series(acc - overall).groupby(items[f]).mean()
        deltas.update(g.to_dict())
    s = pd.Series(deltas).sort_values()
    strict = [SHORT[t] for t in s.index[:2] if s[t] < -0.05]
    relaxed = [SHORT[t] for t in s.index[::-1][:2] if s[t] > 0.05]
    level = acc.mean()
    tone = "Strict" if level < 0.45 else "Careful" if level < 0.6 else "Relaxed" if level > 0.75 else "Balanced"
    name = tone + (f" · no {' or '.join(strict)}" if strict else "") + (f" · OK with {' and '.join(relaxed)}" if relaxed else "")
    allow, cond, deny = (preset_items[i] == 1).mean(), np.isin(preset_items[i], [2, 3, 4]).mean(), (preset_items[i] == 0).mean()
    text = (f"Allows {allow:.0%} of possible data flows, {cond:.0%} only with conditions, and refuses {deny:.0%}. "
            + (f"Stricter than most households about {', '.join(strict)}" if strict else "Close to most households")
            + (f"; more open about {', '.join(relaxed)}." if relaxed else "."))
    return name, text

## Build the files

In [5]:
def leaves(node):
    return [node["preset"]] if "preset" in node else [p for b in node["branches"] for p in leaves(b["next"])]
reached = set(leaves(tree["root"]))
n = len(D)

out_presets = []
for i, p in enumerate(presets["presets"]):
    name, text = describe(i)
    rules = []
    for j, r in enumerate(p["rules"]):
        rule = {"id": f"{p['id']}-r{j}", "effect": r["effect"], "condition": r["condition"],
                "scope": r["scope"], "enabled": True, "origin": {"kind": "preset", "preset_id": p["id"]}}
        rule["title"] = title(rule)
        rules.append(rule)
    out_presets.append({"id": p["id"], "name": f"{tier(i)} · {PRESET_NAMES[p['id']]}" if p["id"] in PRESET_NAMES else name,
                        "tier": tier(i), "description": text,
                        "households_share": round(p["households"] / n, 3), "reached_by_onboarding": p["id"] in reached,
                        "rules": rules})

source = {"generated": date.today().isoformat(), "by": "ppd-household-presets notebooks 01-07",
          "data": "synthetic households (not survey data); see data/DATACARD.md",
          "k": K, "depth": tree["depth"], "questions_avg": tree["questions_avg"], "questions_max": tree["questions_max"],
          "match_real_device_flows_cv": tree["match_real_cv"], "one_preset_for_everyone_cv": tree["one_preset_real_cv"]}

presets_doc = {
    "version": "1.0.0", "source": source,
    "rule_format": ("Like the app's Rule (shared/types.ts): scope lists core ppd: terms per field (a missing field means any), "
                    "effect allow or dont_allow, strictest matching rule wins. New: 'condition' on an allow rule is a PPD qualifier "
                    "the compiler must send as constraints (protocol-04 7.11-7.12); origin kind 'preset'. "
                    "Exactly one rule of a preset matches any flow."),
    "presets": out_presets,
}

def prune(node):
    # Drop questions whose answers all lead to the same preset: nobody should answer a question that changes nothing.
    if "preset" in node:
        return {"preset": node["preset"]}
    branches = [{"answers": b["answers"], "next": prune(b["next"])} for b in node["branches"]]
    ends = {json.dumps(b["next"], sort_keys=True) for b in branches}
    return branches[0]["next"] if len(ends) == 1 else {"question": node["question"], "branches": branches}

used = []
def collect(node):
    if "preset" not in node:
        used.append(node["question"])
        for b in node["branches"]:
            collect(b["next"])
root = prune(tree["root"])
collect(root)
onboarding_doc = {
    "version": "2.0.0", "kind": "tree", "source": source,
    "title": "Your household's privacy preset",
    "description": "A few questions place your household in a privacy preset: a short set of rules your devices receive. You can change any rule afterwards.",
    "disclaimer": ("The questions, presets and tree come from synthetic households generated from published IoT privacy studies "
                   "(Emami-Naeini et al. 2017, 2020, 2021) and the method of Smullen et al. 2020. They are not user-tested."),
    "questions": {q: {"prompt": QUESTIONS[q]["text"],
                      "options": [{"id": o, "label": OPTION_LABEL.get(o, o)} for o in QUESTIONS[q]["options"]]}
                  for q in dict.fromkeys(used)},
    "walk": ("Show the root question; follow the branch whose 'answers' contain the chosen option; repeat until a node has "
             "'preset'. A question can appear again further down a path with a finer split: reuse the earlier answer, "
             "never ask twice."),
    "root": root,
}
print(f"{len(out_presets)} presets ({len(reached)} reached by onboarding), {len(onboarding_doc['questions'])} questions in the tree")

8 presets (8 reached by onboarding), 9 questions in the tree


## Check the exported files

These checks read only the two documents just built, the way the app would.

In [6]:
rank = {0: 0, 4: 1, 3: 2, 2: 3, 1: 4}   # strictest first: deny, then the narrower conditions, then allow

def apply_preset(rules):
    # The app's semantics: a rule matches when every listed field contains the flow's term; the strictest match wins.
    out = np.full(len(items), -1)
    for r in rules:
        inside = np.ones(len(items), dtype=bool)
        for f, terms in r["scope"].items():
            inside &= items[f].isin(terms).to_numpy()
        d = decision_of(r)
        better = inside & ((out == -1) | (np.vectorize(rank.get)(np.where(out == -1, 1, out)) > rank[d]))
        out[better] = d
    return out

def asked_on_path(node, row):
    # Distinct questions a household is actually asked (repeats reuse the earlier answer).
    seen = []
    while "preset" not in node:
        if node["question"] not in seen:
            seen.append(node["question"])
        node = next(b["next"] for b in node["branches"] if row[node["question"]] in b["answers"])
    return len(seen)

def walk(node, row):
    while "preset" not in node:
        node = next(b["next"] for b in node["branches"] if row[node["question"]] in b["answers"])
    return node["preset"]

ids = [p["id"] for p in out_presets]
applied = np.stack([apply_preset(p["rules"]) for p in out_presets])
chosen = np.array([ids.index(walk(onboarding_doc["root"], row)) for row in answers.to_dict("records")])
def branches_ok(node):
    if "preset" in node:
        return node["preset"] in ids
    opts = [o["id"] for o in onboarding_doc["questions"][node["question"]]["options"]]
    covered = [a for b in node["branches"] for a in b["answers"]]
    return sorted(covered) == sorted(opts) and all(branches_ok(b["next"]) for b in node["branches"])

checks = {
    "exported rules give exactly each preset's decisions on all items": (applied == preset_items).all(),
    "exactly one rule matches every flow": all((sum(np.ones(len(items), dtype=bool) & np.logical_and.reduce(
        [items[f].isin(t).to_numpy() for f, t in r["scope"].items()] or [np.ones(len(items), dtype=bool)]) for r in p["rules"]) == 1).all()
        for p in out_presets),
    "every term is a core ppd: term of its field": all(set(t) <= set(TERMS[f]) for p in out_presets for r in p["rules"] for f, t in r["scope"].items()),
    "in-home / on-device conditions only on use and inference": all(
        set(r["scope"].get("action", TERMS["action"])) <= {"ppd:use", "ppd:inference"}
        for p in out_presets for r in p["rules"] if (r["condition"] or "").startswith("processing_boundary")),
    "at most 15 rules per preset": all(len(p["rules"]) <= 15 for p in out_presets),
    "every tree branch covers each answer exactly once and ends in a known preset": branches_ok(onboarding_doc["root"]),
    "walking the tree gives every household a preset": len(chosen) == n,
}
assert all(checks.values()), checks
match_real = (preset_items[chosen][:, real] == D[:, real]).mean()
asked = np.array([asked_on_path(onboarding_doc["root"], row) for row in answers.to_dict("records")])
print(f"walking the exported tree: households' own decisions on real-device flows matched {match_real:.1%} (in-sample); "
      f"questions asked: {asked.mean():.1f} on average, at most {asked.max()}")
pd.Series(checks, name="holds")

walking the exported tree: households' own decisions on real-device flows matched 85.0% (in-sample); questions asked: 4.5 on average, at most 5


exported rules give exactly each preset's decisions on all items                True
exactly one rule matches every flow                                             True
every term is a core ppd: term of its field                                     True
in-home / on-device conditions only on use and inference                        True
at most 15 rules per preset                                                     True
every tree branch covers each answer exactly once and ends in a known preset    True
walking the tree gives every household a preset                                 True
Name: holds, dtype: bool

## Preview

In [7]:
pd.DataFrame([{"preset": p["id"], "name": p["name"], "households": f"{p['households_share']:.0%}",
               "onboarding reaches it": p["reached_by_onboarding"], "rules": len(p["rules"])} for p in out_presets]).set_index("preset")

,name,households,onboarding reaches it,rules
preset,,,,
k8-p0,Privacy-tightened · Keep it at home,21%,True,15
k8-p1,Balanced · No personalization,4%,True,15
k8-p2,Balanced · Device maker only,15%,True,15
k8-p3,"Balanced · Ads OK, no analytics",6%,True,15
k8-p4,Cloud-assisted · Ads and analytics OK,19%,True,15
k8-p5,Cloud-assisted · No ads,8%,True,14
k8-p6,"Balanced · No ads, no analytics",5%,True,15
k8-p7,"Balanced · No ads, careful with sharing",23%,True,15


In [8]:
first = next(p for p in out_presets if p["reached_by_onboarding"])
print(f"{first['name']}\n{first['description']}\n")
for r in first["rules"]:
    print(" ", r["title"])

Privacy-tightened · Keep it at home
Allows 45% of possible data flows, 9% only with conditions, and refuses 46%. Stricter than most households about other companies, usage data; more open about keeping it home.

  Never: any data except device and network details · made public or going to other companies · except when used
  Allow: staying in your home
  Allow only if processed in your home: any data except device and network details · made public or going to other companies · for anything except advertising · when used
  Never: any data except device and network details · going to the device maker · for anything except making features work, security · when used to work out new information or sent on
  Allow: device and network details · leaving your home · for anything except advertising, improving products · except when used
  Allow: going to the device maker · for anything except advertising · when used
  Allow: any data except device and network details · going to the device maker 

In [9]:
def show(node, indent=""):
    if "preset" in node:
        print(f"{indent}→ {next(p['name'] for p in out_presets if p['id'] == node['preset'])}")
        return
    q = onboarding_doc["questions"][node["question"]]
    print(f"{indent}{q['prompt']}")
    labels = {o["id"]: o["label"] for o in q["options"]}
    for b in node["branches"]:
        print(f"{indent}  [{' / '.join(labels[a] for a in b['answers'])}]")
        show(b["next"], indent + "    ")
show(onboarding_doc["root"])

Would you let a smart device use who is home and readings from your home to personalize how it works, without the data leaving your home?
  [No]
    May companies use your data for: personalizing what you see?
      [Never / It depends]
        Would you let a device send how you use it to its maker to improve their products?
          [No / Only with limits]
            How you use the device: may it leave your home?
              [Never / It depends]
                Would you let a device send video or audio from your home to its maker's cloud, so you can use it remotely?
                  [No / Only with limits]
                    → Privacy-tightened · Keep it at home
                  [Yes]
                    → Balanced · No personalization
              [That's fine]
                Would you let a device's maker use a profile of how you use it to show you ads?
                  [No]
                    → Balanced · No personalization
                  [Only with limits / Yes]
 

## Save

In [10]:
EXPORT.mkdir(exist_ok=True)
(EXPORT / "presets.json").write_text(json.dumps(presets_doc, indent=2, ensure_ascii=False) + "\n")
(EXPORT / "onboarding.json").write_text(json.dumps(onboarding_doc, indent=2, ensure_ascii=False) + "\n")
print("wrote", [p.name for p in sorted(EXPORT.iterdir())])

wrote ['onboarding-tree.svg', 'onboarding.json', 'presets.json']


## Picture of the onboarding tree

`export/onboarding-tree.svg`, for the README: questions left to right, the answers on the arrows, and the preset each
path ends in. Presets carry their rank from the tightest privacy (1, refuses the most flows) to the loosest (8), and
the colour of their CableLabs tier.

In [11]:
from textwrap import wrap
from xml.sax.saxutils import escape

SHORT_Q = {   # a few words per question for the picture; the full wording is in onboarding.json and the README
    "real_personalized_heating": "Personalize from readings in your home (data stays home)?",
    "purpose_personal": "Your data for personalizing what you see?",
    "real_usage_improve": "Send how you use it to the maker to improve products?",
    "data_usage": "May how you use the device leave your home?",
    "real_video_cloud": "Video or audio to the maker's cloud?",
    "real_profile_ads": "Maker uses a profile of your use for ads?",
    "concern": "How much do you worry? (1-5)",
    "purpose_analytics": "Your data for improving products?",
    "real_viewing_advertisers": "Maker shares how you use it with advertisers?",
}
SHORT_A = {"Only with limits": "Limits", "That's fine": "Fine", "It depends": "Depends", "1 · Not at all": "1", "5 · A lot": "5"}   # answers, shortened to fit the arrows
TIER_COLOUR = {"Privacy-tightened": ("#FCE8E6", "#B3261E"), "Balanced": ("#E8F0FE", "#0B57D0"), "Cloud-assisted": ("#E6F4EA", "#1E7D34")}

# Rank: most refused flows first; ties go to the preset with more conditions.
refuse = [(preset_items[i] == 0).mean() for i in range(len(out_presets))]
cond = [(preset_items[i] >= 2).mean() for i in range(len(out_presets))]
order = sorted(range(len(out_presets)), key=lambda i: (-refuse[i], -cond[i]))
RANK = {out_presets[i]["id"]: r + 1 for r, i in enumerate(order)}
BY_ID = {p["id"]: p for p in out_presets}

COL, ROW, QW, LW, TOP, LEFT = 370, 50, 210, 290, 95, 20   # a 160 px gap between columns leaves room for the answers
def depth(n):
    return 0 if "preset" in n else 1 + max(depth(b["next"]) for b in n["branches"])
DEPTH = depth(root)
nodes, edges, row = [], [], [0]
def place(n, d):
    if "preset" in n:
        y = TOP + row[0] * ROW
        row[0] += 1
        nodes.append(("leaf", n["preset"], LEFT + DEPTH * COL, y))
        return y
    labels = {o["id"]: SHORT_A.get(o["label"], o["label"]) for o in onboarding_doc["questions"][n["question"]]["options"]}
    kids = [(place(b["next"], d + 1), " / ".join(labels[a] for a in b["answers"]), "preset" in b["next"]) for b in n["branches"]]
    y = sum(cy for cy, *_ in kids) / len(kids)
    nodes.append(("q", n["question"], LEFT + d * COL, y))
    edges.extend((d, cy, label, to_leaf, y) for cy, label, to_leaf in kids)
    return y
place(root, 0)

def text(x, y, lines, size=13, weight=400, colour="#1F1F1F", anchor="start"):
    out = []
    y0 = y - (len(lines) - 1) * (size + 2) / 2
    for i, line in enumerate(lines):
        out.append(f'<text x="{x}" y="{y0 + i * (size + 2) + size / 3:.1f}" font-size="{size}" font-weight="{weight}" fill="{colour}" text-anchor="{anchor}">{escape(line)}</text>')
    return "".join(out)

svg = []
for d, cy, label, to_leaf, py in edges:
    x1, x2 = LEFT + d * COL + QW, (LEFT + DEPTH * COL) if to_leaf else LEFT + (d + 1) * COL
    mid = x1 + 30
    svg.append(f'<path d="M{x1},{py:.1f} C{mid},{py:.1f} {mid},{cy:.1f} {mid + 20},{cy:.1f} L{x2},{cy:.1f}" fill="none" stroke="#9AA0A6" stroke-width="1.3"/>')
    svg.append(f'<text x="{mid + 26}" y="{cy - 6:.1f}" font-size="11.5" fill="#3C4043" stroke="#FFFFFF" stroke-width="3" paint-order="stroke">{escape(label)}</text>')
for kind, key, x, y in nodes:
    if kind == "q":
        lines = wrap(SHORT_Q.get(key, onboarding_doc["questions"][key]["prompt"]), 27)
        h = len(lines) * 15 + 14
        svg.append(f'<rect x="{x}" y="{y - h / 2:.1f}" width="{QW}" height="{h}" rx="8" fill="#FFFFFF" stroke="#5F6368" stroke-width="1.2"/>')
        svg.append(text(x + 10, y, lines, 12.5, 500))
    else:
        p = BY_ID[key]
        fill, ink = TIER_COLOUR.get(p["tier"], ("#F1F3F4", "#3C4043"))
        svg.append(f'<rect x="{x}" y="{y - 16}" width="{LW}" height="32" rx="16" fill="{fill}" stroke="{ink}" stroke-width="1.2"/>')
        svg.append(text(x + 14, y, [f"{RANK[key]} · {p['name']}"], 12.5, 600, ink))

W, H = LEFT * 2 + DEPTH * COL + LW, TOP + row[0] * ROW + 10
title = text(LEFT, 30, ["Onboarding: answer at most 5 questions, follow your answer, end in a preset (1 = tightest privacy, 8 = loosest)"], 15, 600) + text(
    LEFT, 54, ["Note: questions and answers are shortened to fit the boxes (Limits = Only with limits, Depends = It depends, Fine = That's fine). "
               "The full wording is in the README and in export/onboarding.json."], 12, 400, "#5F6368")
legend = "".join(f'<rect x="{W - 520 + i * 170}" y="18" width="14" height="14" rx="7" fill="{f}" stroke="{c}"/>' + text(W - 500 + i * 170, 25, [t], 12, 400, c)
                 for i, (t, (f, c)) in enumerate(TIER_COLOUR.items()))
doc = (f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H}" viewBox="0 0 {W} {H}" font-family="Helvetica, Arial, sans-serif">'
       f'<rect width="{W}" height="{H}" fill="#FFFFFF"/>{title}{legend}{"".join(svg)}</svg>')
(EXPORT / "onboarding-tree.svg").write_text(doc + "\n")
assert row[0] == sum(1 for k, *_ in nodes if k == "leaf") and set(RANK) == set(BY_ID)
print(f"wrote onboarding-tree.svg: {sum(1 for k, *_ in nodes if k == 'q')} questions, {row[0]} end points, {W}x{H}")

wrote onboarding-tree.svg: 20 questions, 21 end points, 2180x1155


## For the app

What `ppd-preference-app` needs to use these files:

1. **Rules with a condition.** `Rule` gains `condition?: string` (a qualifier such as `retention=ppd:ephemeral`) on
   allow rules; the compiler sends it as the PPD rule's `constraints` (protocol-04 §7.11–7.12). Rule `origin` gains
   `{ kind: 'preset', preset_id }`.
2. **Onboarding.** A walker for `onboarding.json`: show the root question, follow the branch whose `answers` contain the
   chosen option, repeat until a `preset`; then show the preset's name, description and rules (editable).
3. **Presets.** Load `presets.json`; let the household switch presets later (rules they added themselves stay).
4. **No per-flow questions.** Remove "Needs your decision", the "Ask me first" effect and per-flow decisions (team
   decision following CableLabs' guidance): presets decide every flow.
5. **Verdicts.** The engine compares flows with these rules as today; "allow if …" rules compare the flow's declared
   qualifier with the condition (a flow that declares `ppd:ephemeral` satisfies "only if not kept").